# Full Injection → Detection → Completeness Pipeline

This notebook runs the complete star cluster injection and recovery pipeline:

1. **Load data** from Butler (RSP `deepCoadd`)
2. **Configure** injection parameters
3. **Run injection** using the pipeline
4. **Visualize** injection (summary plots & stamp grids)
5. **Run detection** on the injected image
6. **Show detection results** including MCI diagram
7. **Match** detections to injections (retrieval)
8. **Plot completeness curves** as a function of magnitude

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Ensure pipeline is on the path
pipeline_path = os.path.expanduser('~/INJECT/star-cluster-injection-pipeline')
if pipeline_path not in sys.path:
    sys.path.insert(0, pipeline_path)

from lsst.daf.butler import Butler

from src.config import InjectionConfig, ClusterConfig
from src.pipeline import InjectionPipeline
from src.visualization import (
    plot_injection_summary,
    plot_stamp_grid,
    plot_postage_stamps,
)
from src.detection import run_cluster_detection, plot_detection_diagnostic, plot_mci_diagram
from src.retrieval import ClusterRetrieval, run_source_detection

plt.style.use('tableau-colorblind10')
%matplotlib inline
print('All imports successful!')

---
## 1 · Load Data from Butler

In [ ]:
butler = Butler('dp02', collections='2.2i/runs/DP0.2')

data_id = {'tract': 3828, 'patch': 24, 'band': 'i'}
coadd = butler.get('deepCoadd', dataId=data_id)

# Work on a manageable cutout for the demo
CUTOUT = 1500  # pixels – increase for production runs
image_full = coadd.image.array.copy()
image = image_full[:CUTOUT, :CUTOUT]

print(f'Full coadd shape : {image_full.shape}')
print(f'Cutout shape     : {image.shape}')

fig, ax = plt.subplots(figsize=(7, 7))
vmin, vmax = np.percentile(image, [1, 99])
ax.imshow(image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
ax.set_title('Input deepCoadd cutout')
plt.show()

---
## 2 · Configure Injection Parameters

Edit the values below to customise the injection run.

In [ ]:
cluster_cfg = ClusterConfig(
    profile_type='king',        # 'king', 'plummer', 'eff', 'sersic'
    method='smooth',            # 'smooth' or 'discrete'
    mag_min=20.0,
    mag_max=26.0,
    r_half_min=2.0,             # pixels
    r_half_max=10.0,
    concentration_min=5.0,      # King concentration c = r_t / r_c
    concentration_max=30.0,
)

config = InjectionConfig(
    run_name='demo_full_pipeline',
    n_clusters=100,
    seed=42,
    edge_buffer=50,
    add_noise=True,
    use_actual_psf=True,        # use the coadd PSF from Butler
    save_injected_image=False,
    cluster_config=cluster_cfg,
    tract=data_id['tract'],
    patch=data_id['patch'],
    band=data_id['band'],
    output_dir='outputs/demo_full_pipeline',
)

print(config)

---
## 3 · Run Injection

In [ ]:
pipe = InjectionPipeline(config)

# Load data – pass the exposure so the pipeline can use the real PSF & WCS
pipe.load_data(image=image)  # swap to pipe.load_data(exposure=coadd) for real PSF

# Generate the injection catalog
catalog = pipe.generate_catalog()
print(f'\nFirst catalog entry: {catalog[0]}')

# Run the injection
injected_image, injection_info = pipe.run_injection()
print(f'\nInjected {len(injection_info)} clusters into the image.')

---
## 4 · Visualize the Injection

### 4a – Summary plot (original · injected · difference · distributions)

In [ ]:
plot_injection_summary(
    image,
    injected_image,
    injection_info,
    save_path='outputs/demo_full_pipeline/injection_summary.png',
    show=True,
)

### 4b – Stamp grid (quick look at individual clusters)

In [ ]:
plot_stamp_grid(
    injection_info,
    n_cols=5,
    n_rows=4,
    save_path='outputs/demo_full_pipeline/stamp_grid.png',
    show=True,
)

### 4c – Individual postage stamps (optional – first 3 clusters)

In [ ]:
for info in injection_info[:3]:
    if 'stamps' in info and info['stamps'] is not None:
        plot_postage_stamps(info['stamps'], cluster_id=info.get('id'), show=True)

---
## 5 · Run Detection on the Injected Image

In [ ]:
detections = run_cluster_detection(
    image=injected_image,
    psf_fwhm=3.5,
    threshold_sigma=5.0,
    use_multiscale=True,
    use_mci=True,
    mci_max=0.85,
)

print(f'Detected {len(detections)} candidate sources (extended + point).')

---
## 6 · Detection Results & MCI Diagram

In [ ]:
# MCI diagram: separates extended cluster candidates from point sources
fig_mci = plot_mci_diagram(detections)
plt.show()

In [ ]:
# Overlay detections on the injected image
fig, ax = plt.subplots(figsize=(8, 8))
vmin, vmax = np.percentile(injected_image, [1, 99])
ax.imshow(injected_image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)

det_x = [d['x'] for d in detections]
det_y = [d['y'] for d in detections]
inj_x = [info['x'] for info in injection_info]
inj_y = [info['y'] for info in injection_info]

ax.scatter(inj_x, inj_y, s=80, facecolors='none', edgecolors='lime',
           lw=1.2, label='Injected')
ax.scatter(det_x, det_y, s=40, facecolors='none', edgecolors='red',
           lw=1.0, marker='s', label='Detected')
ax.legend(loc='upper right', fontsize=11)
ax.set_title('Injected vs Detected Sources')
plt.show()

---
## 7 · Match Detections to Injections (Retrieval)

In [ ]:
MATCH_RADIUS = 5.0  # pixels

retrieval = ClusterRetrieval(injection_info, detections)
retrieval.match_detections(match_radius=MATCH_RADIUS)

stats = retrieval.get_summary_statistics()

print('=== Retrieval Summary ===')
print(f"  Injected         : {stats['n_injected']}")
print(f"  Recovered        : {stats['n_recovered']}")
print(f"  Missed           : {stats['n_missed']}")
print(f"  Completeness     : {stats['overall_completeness']:.1%}")
print(f"  50% mag limit    : {stats['mag_50_limit']:.2f}")

---
## 8 · Completeness Curves

In [ ]:
# --- 8a  Completeness vs magnitude ---
mag_bins = np.linspace(config.cluster_config.mag_min,
                       config.cluster_config.mag_max, 15)

completeness_by_mag = retrieval.completeness_vs_magnitude(mag_bins)

fig, ax = plt.subplots(figsize=(8, 5))
bin_centers = 0.5 * (mag_bins[:-1] + mag_bins[1:])
ax.step(bin_centers, completeness_by_mag, where='mid', lw=2, color='royalblue')
ax.axhline(0.5, ls='--', color='gray', label='50 %')
ax.axhline(0.9, ls=':', color='gray', label='90 %')
ax.set_xlabel('Injected Magnitude', fontsize=13)
ax.set_ylabel('Completeness', fontsize=13)
ax.set_ylim(-0.05, 1.05)
ax.set_title('Completeness vs Magnitude')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('outputs/demo_full_pipeline/completeness_vs_mag.png', dpi=150)
plt.show()

In [ ]:
# --- 8b  Completeness vs half-light radius ---
rh_bins = np.linspace(config.cluster_config.r_half_min,
                      config.cluster_config.r_half_max, 10)

completeness_by_rh = retrieval.completeness_vs_size(rh_bins)

fig, ax = plt.subplots(figsize=(8, 5))
bin_centers_rh = 0.5 * (rh_bins[:-1] + rh_bins[1:])
ax.step(bin_centers_rh, completeness_by_rh, where='mid', lw=2, color='darkorange')
ax.axhline(0.5, ls='--', color='gray', label='50 %')
ax.set_xlabel('Half-light Radius (pixels)', fontsize=13)
ax.set_ylabel('Completeness', fontsize=13)
ax.set_ylim(-0.05, 1.05)
ax.set_title('Completeness vs Size')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('outputs/demo_full_pipeline/completeness_vs_size.png', dpi=150)
plt.show()

In [ ]:
# --- 8c  2-D completeness map (magnitude × size) ---
completeness_2d = retrieval.completeness_2d(mag_bins, rh_bins)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.pcolormesh(mag_bins, rh_bins, completeness_2d.T,
                   cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Completeness')
ax.set_xlabel('Magnitude', fontsize=13)
ax.set_ylabel('Half-light Radius (pixels)', fontsize=13)
ax.set_title('2-D Completeness Map')
plt.tight_layout()
plt.savefig('outputs/demo_full_pipeline/completeness_2d.png', dpi=150)
plt.show()

---
## 9 · Save Results

In [ ]:
pipe.retrieval = retrieval
pipe.detection_catalog = detections
pipe.save_results()

print('\nDone – all outputs saved to:', config.output_dir)